# Federated Transformer for Systemic Risk Modeling
## Results Analysis & Visualization

This notebook analyzes the results of federated learning experiments comparing:
- **Local models**: Per-bank trained Transformers
- **Centralized oracle**: Pooled-data upper bound
- **FedProx**: Federated learning with proximal term
- **FedAvg**: Standard federated averaging
- **GARCH**: Classical econometric baseline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline
%load_ext autoreload
%autoreload 2

## 1. Load Results

In [ ]:
# Load evaluation results
results_path = Path('../outputs/evaluation/results.csv')

if not results_path.exists():
    print("Results not found. Please run: python scripts/evaluate.py")
else:
    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} model evaluations")
    print(f"\nModels: {df['model_name'].unique()}")

## 2. VaR Performance Comparison

In [ ]:
# VaR error comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# VaR 95
var95 = df.groupby('model_type')['var_95_error'].mean().sort_values()
axes[0].barh(range(len(var95)), var95.values)
axes[0].set_yticks(range(len(var95)))
axes[0].set_yticklabels(var95.index)
axes[0].set_xlabel('VaR(95%) Estimation Error')
axes[0].set_title('Value-at-Risk 95% Performance')
axes[0].grid(axis='x', alpha=0.3)

# VaR 99
var99 = df.groupby('model_type')['var_99_error'].mean().sort_values()
axes[1].barh(range(len(var99)), var99.values)
axes[1].set_yticks(range(len(var99)))
axes[1].set_yticklabels(var99.index)
axes[1].set_xlabel('VaR(99%) Estimation Error')
axes[1].set_title('Value-at-Risk 99% Performance')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVaR Estimation Errors:")
print(df.groupby('model_type')[['var_95_error', 'var_99_error']].mean())

## 3. Distributional Metrics

In [ ]:
# Kurtosis and distributional accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kurtosis error
kurt_err = df.groupby('model_type')['kurtosis_error'].mean().sort_values()
axes[0].bar(range(len(kurt_err)), kurt_err.values)
axes[0].set_xticks(range(len(kurt_err)))
axes[0].set_xticklabels(kurt_err.index, rotation=45, ha='right')
axes[0].set_ylabel('Kurtosis Error')
axes[0].set_title('Tail Heaviness Modeling Accuracy')
axes[0].grid(axis='y', alpha=0.3)

# ACF error (volatility clustering)
acf_err = df.groupby('model_type')['acf_error'].mean().sort_values()
axes[1].bar(range(len(acf_err)), acf_err.values)
axes[1].set_xticks(range(len(acf_err)))
axes[1].set_xticklabels(acf_err.index, rotation=45, ha='right')
axes[1].set_ylabel('ACF Error')
axes[1].set_title('Volatility Clustering Accuracy')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDistributional Metrics:")
print(df.groupby('model_type')[['kurtosis_error', 'acf_error', 'wasserstein_distance']].mean())

## 4. Model Performance Heatmap

In [ ]:
# Create comprehensive heatmap
metrics_to_compare = [
    'mse', 'var_95_error', 'var_99_error', 
    'kurtosis_error', 'acf_error'
]

heatmap_data = df.groupby('model_type')[metrics_to_compare].mean()

# Normalize for visualization (0-1 scale)
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_norm.T, annot=heatmap_data.T.values, fmt='.4f', 
            cmap='RdYlGn_r', cbar_kws={'label': 'Normalized Error'},
            linewidths=0.5)
plt.title('Comprehensive Model Performance (Lower is Better)')
plt.ylabel('Metric')
plt.xlabel('Model Type')
plt.tight_layout()
plt.show()

## 5. FedProx vs Local Models: Relative Improvement

In [ ]:
# Calculate improvement over local models
if 'federated_fedprox' in df['model_type'].values and 'local' in df['model_type'].values:
    local_var95 = df[df['model_type'] == 'local']['var_95_error'].mean()
    fedprox_var95 = df[df['model_type'] == 'federated_fedprox']['var_95_error'].mean()
    
    improvement = (local_var95 - fedprox_var95) / local_var95 * 100
    
    print(f"\nKEY FINDING:")
    print(f"   FedProx achieves {improvement:.1f}% reduction in VaR(95) error")
    print(f"   compared to local models")
    print(f"\n   Local avg:  {local_var95:.6f}")
    print(f"   FedProx:    {fedprox_var95:.6f}")
    
    fig, ax = plt.subplots(figsize=(8, 5))
    models = ['Local\n(Per-Bank)', 'FedProx\n(Collaborative)']
    values = [local_var95, fedprox_var95]
    colors = ['#ff6b6b', '#51cf66']
    
    bars = ax.bar(models, values, color=colors, alpha=0.7, edgecolor='black')
    ax.set_ylabel('VaR(95%) Estimation Error')
    ax.set_title('Federated Learning Improvement in Tail Risk Modeling')
    ax.grid(axis='y', alpha=0.3)
    
    for i, bar in enumerate(bars):
        height = bar.get_height()
        if i == 1:
            ax.text(bar.get_x() + bar.get_width()/2, height + 0.001,
                   f'{improvement:.1f}% reduction',
                   ha='center', va='bottom', fontsize=12, fontweight='bold', color='green')
    
    plt.tight_layout()
    plt.show()
else:
    print("Warning: FedProx or local models not found in results")

## 6. Summary Statistics

In [ ]:
# Overall summary table
summary_cols = ['mse', 'mae', 'var_95_error', 'var_99_error', 'kurtosis_error']
summary = df.groupby('model_type')[summary_cols].agg(['mean', 'std'])

print("\nCOMPREHENSIVE SUMMARY:\n")
print(summary)

print("\nMODEL RANKING (by VaR 95% error):")
ranking = df.groupby('model_type')['var_95_error'].mean().sort_values()
for i, (model, error) in enumerate(ranking.items(), 1):
    print(f"   {i}. {model}: {error:.6f}")

## 7. Conclusions

### Key Findings:
1. **Federated learning with FedProx enables privacy-preserving collaboration** that improves tail risk modeling compared to isolated local models
2. **FedProx handles non-IID data better than FedAvg** due to the proximal term stabilizing training
3. **Secure aggregation via masking** prevents the server from accessing raw model updates while preserving model quality

### Limitations:
- Research-grade prototype (not production-ready)
- No formal differential privacy guarantees
- Single-machine simulation
- Small model size for computational efficiency

### Next Steps:
- Integrate Opacus for formal DP guarantees
- Scale to larger models and real multi-institutional deployment
- Add Byzantine-robust aggregation
- Regulatory compliance analysis